# Promptle Style-LoRA — train + evaluate

End-to-end: configure → build a per-style dataset → train its LoRA → compare against the
current prompt-suffix baseline → evaluate. The last cell trains all 12 styles.

**Why LoRA here:** Promptle content is open-ended, but art *style* is a closed set of 12
(`ArtStyleSeeder.java`). We move style from a 'loud' prompt suffix into adapter weights, so
the content prompt stays clean and the LoRA scale becomes the style knob.

Setup once: `pip install -e '.[eval,notebook]'` from the `promptle-diffusion/` dir, and set
`PROMPTLE_SD_MODEL` to an SD 1.5 HF id or local diffusers folder for training.

In [ ]:
import matplotlib.pyplot as plt
from promptle_diffusion.config import Config, DEFAULT_LORA_SCALE
from promptle_diffusion.styles import STYLES, by_slug

cfg = Config()
cfg.ensure_dirs()
print('device:', cfg.device, '| dtype:', cfg.dtype, '| model:', cfg.model_id)
print('styles:', [s.slug for s in STYLES])

## 1. Build a dataset for one style

Pick a style. `curated` styles pull public-domain art from the Met API; `synthetic` styles
bootstrap from SD 1.5. After `fetch`, **manually cull** bad images in `datasets/<slug>/raw/`,
then `prepare` resizes survivors and `caption` writes `metadata.jsonl`.

In [ ]:
from promptle_diffusion.data import build_dataset as bd
from promptle_diffusion.data.caption import Blip, caption_style

style = by_slug('watercolor')
print('source:', style.source, '| trigger:', style.trigger)

bd.fetch(style, cfg, limit=120)
# --- manually cull datasets/<slug>/raw/ here, then continue ---
bd.prepare(style, cfg)
caption_style(style, cfg, Blip(device=cfg.device))

## 2. Train its LoRA

~30–60 min on a single 12–16GB GPU. Output: `lora/<slug>.safetensors`.

In [ ]:
from promptle_diffusion.train_lora import TrainConfig, train_style

train_style(style, cfg, TrainConfig(rank=16, lr=1e-4, max_steps=1200))

## 3. Comparison grid

Same prompt + seed across: no style | prompt-suffix baseline | trained LoRA.

In [ ]:
from promptle_diffusion.pipeline import PromptleSD

sd = PromptleSD(cfg)
prompts = ['a fox in a meadow', 'a castle on a hill at dawn', 'a teacup on a saucer']
scale = DEFAULT_LORA_SCALE
cols = ['base', 'suffix (current)', 'lora (ours)']

fig, axes = plt.subplots(len(prompts), 3, figsize=(9, 3 * len(prompts)))
for r, p in enumerate(prompts):
    imgs = [
        sd.generate_t2i(p, style=None, seed=7 + r),
        sd.generate_t2i(f'{p}, ({style.trigger} style:{scale})', style=None, seed=7 + r),
        sd.generate_t2i(p, style=style, scale=scale, seed=7 + r),
    ]
    for c, im in enumerate(imgs):
        ax = axes[r][c] if len(prompts) > 1 else axes[c]
        ax.imshow(im); ax.axis('off')
        if r == 0: ax.set_title(cols[c])
plt.tight_layout(); plt.show()

## 4. Evaluate (CLIP-style score + FID vs reference set)

In [ ]:
from promptle_diffusion.eval import evaluate_style

evaluate_style(style, cfg)

## 5. Train all 12 styles

Heavy. Assumes each style's dataset is already built + captioned.

In [ ]:
for s in STYLES:
    meta = cfg.dataset_dir / s.slug / 'images' / 'metadata.jsonl'
    if not meta.exists():
        print('skip (no dataset):', s.slug); continue
    train_style(s, cfg, TrainConfig(max_steps=1200))